# Text Classification (Sentiment)

## Setup

### Import Dependencies

In [8]:
import pandas as pd
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

### Load CSV file into DataFrame

In [9]:
load_dotenv()

df = pd.read_csv("../data/transformed_news_data.csv")

df.head()

,id,published_date,title,source,tags,crawl_date,tickers,day_name
0,94624618,2026-04-19,Prediction: It's Not Too Late to Buy Broadcom ...,finance.yahoo.com,"['Broadcom', 'Broadcom Stock', 'Communication ...",2026-04-19,"['avgo', 'brcm', 'intc', 'iqnt', 'meta', 'nflx...",Sunday
1,94622595,2026-04-19,This Stock Will Be More Profitable Than Amazon...,finance.yahoo.com,"['Amazon', 'Communication Services', 'Compound...",2026-04-19,"['amd', 'amzn', 'intc', 'iqnt', 'meta', 'mu', ...",Sunday
2,94621400,2026-04-19,Option Traders Chasing Torrid Stock Rally Turn...,bloomberg.com,"['Financial Services', 'Stock', 'Technology', ...",2026-04-19,"['jpm', 'meta', 'msft']",Sunday
3,94612855,2026-04-18,Broadcom (AVGO) Ranks Among Unrivaled Stocks o...,finance.yahoo.com,"['Blue Chip Stocks', 'Broadcom Inc.', 'Inc.', ...",2026-04-18,"['avgo', 'brcm', 'meta']",Saturday
4,94612626,2026-04-18,Meta Platforms (META) Expands AI Chip Deal wit...,insidermonkey.com,"['Stock', 'Technology']",2026-04-18,['meta'],Saturday


## Text Classification (sentiment)

### Run model on news data

In [10]:
client = InferenceClient(
    provider="hf-inference",
    api_key=os.getenv('hugging_face_token'),
)

results = client.text_classification(
    df['title'].to_list(),
    model="ProsusAI/finbert",
)

# Generated with AI (Gemini 3 fast)
df['sentiment_label'] = [res.label for res in results]
df['sentiment_score'] = [res.score for res in results]

df.head()

,id,published_date,title,source,tags,crawl_date,tickers,day_name,sentiment_label,sentiment_score
0,94624618,2026-04-19,Prediction: It's Not Too Late to Buy Broadcom ...,finance.yahoo.com,"['Broadcom', 'Broadcom Stock', 'Communication ...",2026-04-19,"['avgo', 'brcm', 'intc', 'iqnt', 'meta', 'nflx...",Sunday,neutral,0.810057
1,94622595,2026-04-19,This Stock Will Be More Profitable Than Amazon...,finance.yahoo.com,"['Amazon', 'Communication Services', 'Compound...",2026-04-19,"['amd', 'amzn', 'intc', 'iqnt', 'meta', 'mu', ...",Sunday,positive,0.585464
2,94621400,2026-04-19,Option Traders Chasing Torrid Stock Rally Turn...,bloomberg.com,"['Financial Services', 'Stock', 'Technology', ...",2026-04-19,"['jpm', 'meta', 'msft']",Sunday,negative,0.485695
3,94612855,2026-04-18,Broadcom (AVGO) Ranks Among Unrivaled Stocks o...,finance.yahoo.com,"['Blue Chip Stocks', 'Broadcom Inc.', 'Inc.', ...",2026-04-18,"['avgo', 'brcm', 'meta']",Saturday,positive,0.770807
4,94612626,2026-04-18,Meta Platforms (META) Expands AI Chip Deal wit...,insidermonkey.com,"['Stock', 'Technology']",2026-04-18,['meta'],Saturday,positive,0.745573


In [11]:
# Assign numerical labels to each sentiment
df['numerical_sentiment'] = df.apply(lambda x: \
                                     1 if x['sentiment_label'] == 'positive'
                                     else (0 if x['sentiment_label'] == 'neutral' else -1),
                                     axis=1)

# Flag if an article is positive, negative or neutral
df['is_positive'] = df.apply(lambda x: 1 if x['sentiment_label'] == 'positive' else 0, axis=1)
df['is_negative'] = df.apply(lambda x: 1 if x['sentiment_label'] == 'negative' else 0, axis=1)
df['is_neutral'] = df.apply(lambda x: 1 if x['sentiment_label'] == 'neutral' else 0, axis=1)

df.head()

,id,published_date,title,source,tags,crawl_date,tickers,day_name,sentiment_label,sentiment_score,numerical_sentiment,is_positive,is_negative,is_neutral
0,94624618,2026-04-19,Prediction: It's Not Too Late to Buy Broadcom ...,finance.yahoo.com,"['Broadcom', 'Broadcom Stock', 'Communication ...",2026-04-19,"['avgo', 'brcm', 'intc', 'iqnt', 'meta', 'nflx...",Sunday,neutral,0.810057,0,0,0,1
1,94622595,2026-04-19,This Stock Will Be More Profitable Than Amazon...,finance.yahoo.com,"['Amazon', 'Communication Services', 'Compound...",2026-04-19,"['amd', 'amzn', 'intc', 'iqnt', 'meta', 'mu', ...",Sunday,positive,0.585464,1,1,0,0
2,94621400,2026-04-19,Option Traders Chasing Torrid Stock Rally Turn...,bloomberg.com,"['Financial Services', 'Stock', 'Technology', ...",2026-04-19,"['jpm', 'meta', 'msft']",Sunday,negative,0.485695,-1,0,1,0
3,94612855,2026-04-18,Broadcom (AVGO) Ranks Among Unrivaled Stocks o...,finance.yahoo.com,"['Blue Chip Stocks', 'Broadcom Inc.', 'Inc.', ...",2026-04-18,"['avgo', 'brcm', 'meta']",Saturday,positive,0.770807,1,1,0,0
4,94612626,2026-04-18,Meta Platforms (META) Expands AI Chip Deal wit...,insidermonkey.com,"['Stock', 'Technology']",2026-04-18,['meta'],Saturday,positive,0.745573,1,1,0,0


### Calculate Metrics

In [12]:
# Calculate metrics
metrics_df = df.groupby(['published_date'])\
                .agg({
                'numerical_sentiment': 'mean',
                'sentiment_score': 'mean',
                'is_positive': 'sum',
                'is_negative': 'sum',
                'is_neutral': 'sum',
                'sentiment_label': 'count'})\
                .reset_index()\
                .rename(
                columns={'published_date': 'date',
                         'is_positive': 'positive_count',
                         'is_negative': 'negative_count',
                         'is_neutral': 'neutral_count',
                         'sentiment_label': 'total_articles',
                         'numerical': 'mean_sentiment',
                         'sentiment_score': 'mean_sentiment_probability'})

# Calculate Percentages
metrics_df['percent_positive'] = metrics_df['positive_count'] / metrics_df['total_articles']
metrics_df['percent_negative'] = metrics_df['negative_count'] / metrics_df['total_articles']
metrics_df['percent_neutral'] = metrics_df['neutral_count'] / metrics_df['total_articles']

# Remove count columns use the percentages of totals instead
metrics_df = metrics_df.drop(['positive_count', 'negative_count', 'neutral_count', 'total_articles'], axis=1)

metrics_df.head()

,date,numerical_sentiment,mean_sentiment_probability,percent_positive,percent_negative,percent_neutral
0,2026-04-14,0.076923,0.794120,0.230769,0.153846,0.615385
1,2026-04-15,0.000000,0.786592,0.242424,0.242424,0.515152
2,2026-04-16,0.333333,0.783817,0.333333,0.000000,0.666667
3,2026-04-17,0.062500,0.786713,0.187500,0.125000,0.687500
4,2026-04-18,0.090909,0.824430,0.272727,0.181818,0.545455


## Write output DataFrame as CSV File

In [13]:
# Output as CSV
metrics_df.to_csv('../data/article_metrics.csv', index=False)